In [1]:
# Встановлення кодека напряму з GitHub (фікс помилки PyPI)
!pip install -q git+https://github.com/lucadellalib/focalcodec.git

# Встановлення всіх інших залежностей
!pip install -q lightning openai-whisper jiwer speechbrain soundfile torchaudio transformers pandas tqdm librosa


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.3 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 7.8 MB/s eta 0:00:0000:0100:01mm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 40.1 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import librosa
from tqdm.auto import tqdm
from dataclasses import dataclass
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from torch.utils.data import Dataset, DataLoader
import lightning as L

L.seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Seed set to 42


cell 3: dataset aquisition

In [ ]:
root_dir = "./data"
os.makedirs(root_dir, exist_ok=True)

# Download and extract the dataset
ljspeech_dataset = torchaudio.datasets.LJSPEECH(root=root_dir, download=True)

meta = pd.read_csv(f'{root_dir}/LJSpeech-1.1/metadata.csv', sep='|', header=None,
                   names=['id', 'text', 'text_norm'])
meta['text_norm'] = meta['text_norm'].fillna(meta['text'])
meta['filepath'] = meta['id'].apply(lambda s: f'{root_dir}/LJSpeech-1.1/wavs/{s}.wav')

# Train/Val split
rng = np.random.default_rng(42)
perm = rng.permutation(len(meta))
split_at = int(0.9 * len(meta))
train_df = meta.iloc[perm[:split_at]].reset_index(drop=True)
val_df   = meta.iloc[perm[split_at:]].reset_index(drop=True)

100%|██████████| 2.56G/2.56G [00:12<00:00, 213MB/s] 


4. audio codec

In [ ]:
from focalcodec import FocalCodec

codec = FocalCodec.from_pretrained('lucadellalib/focalcodec_25hz').to(device).eval()
for p in codec.parameters(): 
    p.requires_grad_(False)

CODEBOOK_SIZE = int(codec.codebook.shape[0])
CODEC_SR = codec.sample_rate_input
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

def load_wav_16k(path):
    arr, sr = sf.read(path, dtype='float32', always_2d=False)
    if arr.ndim > 1: arr = arr.mean(-1)
    wav = torch.from_numpy(arr)
    if sr != CODEC_SR: wav = torchaudio.functional.resample(wav, sr, CODEC_SR)
    return wav

In [ ]:
import matplotlib.pyplot as plt
import librosa.display
from IPython.display import Audio, display

def visualize_results(path, label="Original"):
    # Load and process
    wav = load_wav_16k(path).to(device)
    
    with torch.no_grad():
        # Encode to tokens
        tokens = codec.sig_to_toks(wav.unsqueeze(0))
        # Decode back to audio
        reconstructed = codec.toks_to_sig(tokens).squeeze(0).cpu().numpy()
    
    original_np = wav.cpu().numpy()
    
    # Plotting
    fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    
    # Original Spectrogram
    S_orig = librosa.feature.melspectrogram(y=original_np, sr=CODEC_SR)
    librosa.display.specshow(librosa.power_to_db(S_orig, ref=np.max), ax=ax[0], sr=CODEC_SR, x_axis='time', y_axis='mel')
    ax[0].set_title(f"Original Audio: {os.path.basename(path)}")
    
    # Reconstructed Spectrogram
    S_rec = librosa.feature.melspectrogram(y=reconstructed, sr=CODEC_SR)
    librosa.display.specshow(librosa.power_to_db(S_rec, ref=np.max), ax=ax[1], sr=CODEC_SR, x_axis='time', y_axis='mel')
    ax[1].set_title(f"FocalCodec Reconstruction ({tokens.shape[-1]} tokens)")
    
    plt.tight_layout()
    plt.show()

    print("--- Original Audio ---")
    display(Audio(original_np, rate=CODEC_SR))
    print("--- Reconstructed (Codec) Audio ---")
    display(Audio(reconstructed, rate=CODEC_SR))

# Pick a random sample and check it
sample_row = train_df.iloc[0]
visualize_results(sample_row['filepath'])

: Tokenization and Preprocessing

In [ ]:
CACHE_DIR = os.path.join(root_dir, 'tokenized')
os.makedirs(CACHE_DIR, exist_ok=True)
TRAIN_CACHE = os.path.join(CACHE_DIR, 'train.pt')
VAL_CACHE   = os.path.join(CACHE_DIR, 'val.pt')

@torch.no_grad()
def encode_split(df, cache_path, batch_size=16, overwrite=False, limit=None):
    if os.path.exists(cache_path) and not overwrite:
        return torch.load(cache_path, weights_only=False)

    if limit: df = df.iloc[:limit]

    recs = df.to_dict('records')
    paths = [r['filepath'] for r in recs]
    order = np.argsort([os.path.getsize(p) for p in paths])
    out = [None] * len(recs)
    
    for start in tqdm(range(0, len(recs), batch_size), desc=f"Processing {cache_path}"):
        idxs = order[start:start+batch_size]
        wavs = [load_wav_16k(paths[i]) for i in idxs]
        lens = torch.tensor([w.numel() for w in wavs], dtype=torch.float32)
        L_batch = int(lens.max()); batch = torch.zeros(len(wavs), L_batch)
        for j, w in enumerate(wavs): batch[j, :w.numel()] = w
        batch = batch.to(device)
        toks = codec.sig_to_toks(batch, length=(lens/L_batch).to(device))
        tok_lens = ((lens/L_batch).to(device) * toks.shape[-1]).round().clamp(max=toks.shape[-1]).to(torch.long)
        
        for j, i in enumerate(idxs):
            text_ids = tokenizer.encode(recs[i]['text_norm'], add_special_tokens=False)
            out[i] = {
                'id': recs[i]['id'],
                'text': recs[i]['text_norm'],
                'text_ids': torch.tensor(text_ids, dtype=torch.long),
                'audio_ids': toks[j, :tok_lens[j]].to(torch.long).cpu(),
                'n_text': len(text_ids),
                'n_audio': tok_lens[j].item(),
            }
    torch.save(out, cache_path)
    return out

# Set limit to None for full training
train_items = encode_split(train_df, TRAIN_CACHE, limit=500)
val_items   = encode_split(val_df,   VAL_CACHE, limit=50)

Data loading ultimaties

In [ ]:
class TokenizedLJSpeech(Dataset):
    def __init__(self, cache_path, max_total_len=1024):
        items = torch.load(cache_path, weights_only=False)
        self.items = [it for it in items if it["n_text"] + it["n_audio"] + 2 <= max_total_len]
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    B = len(batch)
    text_lens = torch.tensor([b["text_ids"].numel() for b in batch], dtype=torch.long)
    audio_lens = torch.tensor([b["audio_ids"].numel() for b in batch], dtype=torch.long)
    text_ids = torch.zeros(B, int(text_lens.max()), dtype=torch.long)
    audio_ids = torch.zeros(B, int(audio_lens.max()), dtype=torch.long)
    for i, b in enumerate(batch):
        text_ids[i, : b["text_ids"].numel()] = b["text_ids"]
        audio_ids[i, : b["audio_ids"].numel()] = b["audio_ids"]
    return {"text_ids": text_ids, "text_lens": text_lens, "audio_ids": audio_ids, "audio_lens": audio_lens}

gpt2 tts architecture

In [ ]:
@dataclass
class GPT2TTSConfig:
    base_model: str = "gpt2"
    codebook_size: int = 8192
    n_special_tokens: int = 2
    @property
    def audio_vocab_size(self): return self.codebook_size + self.n_special_tokens
    @property
    def bos_id(self): return self.codebook_size
    @property
    def eos_id(self): return self.codebook_size + 1

class GPT2TTS(nn.Module):
    def __init__(self, cfg: GPT2TTSConfig):
        super().__init__()
        self.cfg = cfg
        self.base = GPT2LMHeadModel.from_pretrained(cfg.base_model)
        H, V = self.base.config.n_embd, cfg.audio_vocab_size
        self.audio_emb = nn.Embedding(V, H)
        self.audio_head = nn.Linear(H, V, bias=False)
        nn.init.normal_(self.audio_emb.weight, std=0.02)
        nn.init.normal_(self.audio_head.weight, std=0.02)

    def _build_inputs(self, text_ids, text_lens, audio_ids, audio_lens):
        B, device = text_ids.size(0), text_ids.device
        L = int((text_lens + audio_lens + 2).max().item())
        inputs = torch.zeros(B, L, self.base.config.n_embd, device=device)
        mask = torch.zeros(B, L, dtype=torch.long, device=device)
        labels = torch.full((B, L), -100, dtype=torch.long, device=device)
        text_emb_all = self.base.transformer.wte(text_ids)

        for i in range(B):
            tl, al = int(text_lens[i].item()), int(audio_lens[i].item())
            inputs[i, :tl] = text_emb_all[i, :tl]
            mask[i, :tl] = 1
            inputs[i, tl] = self.audio_emb.weight[self.cfg.bos_id]
            mask[i, tl] = 1
            inputs[i, tl+1 : tl+1+al] = self.audio_emb(audio_ids[i, :al])
            mask[i, tl+1 : tl+1+al] = 1
            labels[i, tl+1 : tl+1+al] = audio_ids[i, :al]
            inputs[i, tl+al+1] = self.audio_emb.weight[self.cfg.eos_id]
            mask[i, tl+al+1] = 1
            labels[i, tl+al+1] = self.cfg.eos_id
        return inputs, mask, labels

    def forward(self, text_ids, text_lens, audio_ids, audio_lens):
        inputs, mask, labels = self._build_inputs(text_ids, text_lens, audio_ids, audio_lens)
        h = self.base.transformer(inputs_embeds=inputs, attention_mask=mask, return_dict=True).last_hidden_state
        logits = self.audio_head(h)
        shift_logits = logits[:, :-1].float().contiguous()
        shift_labels = labels[:, 1:].contiguous()
        return {"loss": F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100), "logits": logits}

    @torch.no_grad()
    def generate_audio(self, text_ids, max_new_tokens=400, temperature=0.9, top_k=50, top_p=1.0, greedy=False):
        self.eval(); device = text_ids.device
        text_emb = self.base.transformer.wte(text_ids.view(-1).long()).unsqueeze(0)
        bos = self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)
        prefix = torch.cat([text_emb, bos], dim=1)
        out = self.base.transformer(inputs_embeds=prefix, use_cache=True, return_dict=True)
        past, h = out.past_key_values, out.last_hidden_state[:, -1, :]
        generated = []
        for _ in range(max_new_tokens):
            logits = self.audio_head(h).squeeze(0).float()
            if greedy: next_id = int(logits.argmax().item())
            else:
                logits = logits / max(temperature, 1e-8)
                if top_k > 0:
                    kth = torch.topk(logits, min(top_k, logits.size(-1))).values[-1]
                    logits[logits < kth] = -float("inf")
                if 0.0 < top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                    cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_logits[cum_probs - F.softmax(sorted_logits, dim=-1) > top_p] = -float("inf")
                    logits = torch.zeros_like(logits).scatter_(0, sorted_idx, sorted_logits)
                next_id = int(torch.multinomial(F.softmax(logits, dim=-1), 1).item())
            if next_id == self.cfg.eos_id: break
            generated.append(next_id)
            tok_emb = self.audio_emb(torch.tensor([[next_id]], device=device))
            out = self.base.transformer(inputs_embeds=tok_emb, past_key_values=past, use_cache=True, return_dict=True)
            past, h = out.past_key_values, out.last_hidden_state[:, -1, :]
        return torch.tensor(generated, dtype=torch.long, device=device)

training pipeline

In [ ]:
class GPT2TTSLightningModule(L.LightningModule):
    def __init__(self, cfg, lr=3e-4):
        super().__init__(); self.save_hyperparameters()
        self.model = GPT2TTS(cfg); self.lr = lr
    def training_step(self, batch, idx):
        loss = self.model(**batch)["loss"]
        self.log("train_loss", loss, prog_bar=True); return loss
    def validation_step(self, batch, idx):
        loss = self.model(**batch)["loss"]
        self.log("val_loss", loss, prog_bar=True)
    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        return [opt], [{"scheduler": torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs), "interval": "epoch"}]

train_dl = DataLoader(TokenizedLJSpeech(TRAIN_CACHE), batch_size=8, shuffle=True, collate_fn=collate)
val_dl = DataLoader(TokenizedLJSpeech(VAL_CACHE), batch_size=8, collate_fn=collate)

trainer = L.Trainer(max_epochs=3, accelerator="auto", devices=1)
module = GPT2TTSLightningModule(GPT2TTSConfig())
trainer.fit(module, train_dl, val_dl)
trained_model = module.model.to(device)

inference, evaluation

In [ ]:
def get_metrics(wav, sr, text, ref_wav):
    import whisper; from jiwer import cer
    from speechbrain.inference.interfaces import foreign_class
    from speechbrain.inference.speaker import EncoderClassifier

    # CER
    asr = whisper.load_model("base")
    hyp = asr.transcribe(wav.astype(np.float32), fp16=False)["text"].strip().lower()
    cer_val = cer(text.lower(), hyp)
    
    # UTMOS
    utmos_predictor = foreign_class(source="sarulab-speech/UTMOS22", pymodule_file="score.py", classname="UTMOSScore")
    mos_val = utmos_predictor.score(torchaudio.functional.resample(torch.from_numpy(wav).unsqueeze(0), sr, 16000)).item()
    
    # SECS
    secs_classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
    def embed(w): return secs_classifier.encode_batch(torchaudio.functional.resample(torch.from_numpy(w).unsqueeze(0), sr, 16000)).squeeze()
    secs_val = F.cosine_similarity(embed(ref_wav).unsqueeze(0), embed(wav).unsqueeze(0)).item()
    
    return cer_val, mos_val, secs_val

# Run for one sample text
sample_text = "The quick brown fox jumps over the lazy dog."
text_ids = torch.tensor(tokenizer.encode(sample_text, add_special_tokens=False), device=device)
toks = trained_model.generate_audio(text_ids, greedy=True)
wav = codec.toks_to_sig(toks.unsqueeze(0)).squeeze(0).cpu().numpy()
ref_wav = load_wav_16k(train_df.iloc[0]['filepath']).numpy()

cer_v, mos_v, secs_v = get_metrics(wav, CODEC_SR, sample_text, ref_wav)
print(f"Metrics: CER={cer_v:.3f}, UTMOS={mos_v:.3f}, SECS={secs_v:.3f}")